# 실습 1 남극 펭귄 데이터 탐색하기 (학생용)

교과서 78~83쪽

앞에서 살펴본 펭귄의 신체 부위 데이터를 탐색하고, 기계학습 모델에 쓰기 위한 전처리를 해 보자.

| 단계 | 하는 일 |
|---|---|
| 단계 1 데이터 수집하기 | 캐글에서 `penguins_size.csv` 내려받기 |
| 단계 2 데이터 살펴보기 | 불러오기, 속성과 유형, 통계 요약, 시각화 |
| 단계 3 데이터 전처리하기 | 결측치 처리, 이상치 처리, 정규화 |
| 단계 4 핵심 속성 추출하기 | 산점도, 상관계수 히트맵 |

**사용 방법**: 위에서부터 차례로 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자로 속성을 바꿔 가며 다시 실행해 보세요.

In [ ]:
#@title 데이터 준비하기
how = "사이트에서 바로 받기" #@param ["사이트에서 바로 받기", "내 컴퓨터에서 올리기"]
#@markdown 디지털 교과서에 올려 둔 교과서 자료(penguins_size.csv)를 코랩으로 바로 가져옵니다. 직접 모은 파일을 쓰려면 '내 컴퓨터에서 올리기'를 고르세요.
if how.startswith('사이트'):
    import urllib.request
    for name in ['penguins_size.csv']:
        urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/' + name, name)
        print('받았습니다:', name)
else:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
#@title 단계 2 ① 데이터 불러오기 (교과서 79쪽)
import pandas as pd                       # 판다스 라이브러리 가져오기
df = pd.read_csv('penguins_size.csv')     # 데이터 프레임(df) 생성
df.head()                                 # 데이터 프레임 5행 출력

In [ ]:
#@title ② 속성과 유형 살펴보기: df.info()
df.info()

**결과 읽기**: 데이터는 몇 개이고 속성은 몇 개인가요? `Non-Null Count`가 344보다 작은 속성에는 결측치가 있습니다. `Dtype`이 `object`인 속성(species, island, sex)은 범주형, `float64`인 속성은 수치형입니다.

In [ ]:
#@title ③ 통계 정보 확인하기: df.describe()
df.describe()

**결과 읽기**: 부리 깊이의 평균은 약 17, 체질량의 평균은 약 4,200입니다. 속성값의 크기에 차이가 커서 정규화가 필요합니다.

In [ ]:
#@title ④ 데이터 시각화하기: 세 가지 펭귄 종의 데이터 수
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(x=df['species'])            # 막대그래프(countplot) 그리기
plt.show()
print(df['species'].value_counts())

### 데이터 편향성 검토하기
턱끈펭귄(Chinstrap)의 데이터 수가 다른 종보다 적습니다. 이 데이터로 펭귄 종을 예측하는 모델을 만들면 어떤 문제가 생길 수 있을까요?

In [ ]:
#@title 단계 3 ① 결측치 처리하기 (교과서 80쪽)
print('속성별 결측치 수')
print(df.isnull().sum())

df = df.dropna(axis=0)                    # 결측치가 있는 행 삭제
print()
print('결측치를 지운 뒤 데이터 크기:', df.shape)

In [ ]:
#@title ② 이상치 확인하기: 상자그림 (교과서 81쪽)
attribute = "culmen_length_mm" #@param ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
by_species = False #@param {type:"boolean"}

plt.figure(figsize=(7, 2.5 if not by_species else 3.5))
if by_species:
    sns.boxplot(data=df, x=attribute, y='species')
else:
    sns.boxplot(data=df, x=attribute)
plt.show()

# 상자그림에서 점으로 나타나는 값(이상치)을 직접 계산해 보기
Q1, Q3 = df[attribute].quantile(0.25), df[attribute].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = df[(df[attribute] < lower) | (df[attribute] > upper)]
print(f'Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}')
print(f'이상치 기준: {lower:.2f} 미만 또는 {upper:.2f} 초과 → 전체 데이터에서 이상치 {len(outliers)}개')

속성을 바꿔 가며 실행해 보세요. 전체 데이터로는 이상치가 없어 보여도, **종별로** 나누어 보면 이상치가 보이는 속성이 있습니다(예: 턱끈펭귄의 체질량, 교과서 75쪽).

In [ ]:
#@title ③ 범주형 속성의 이상치 확인하기: unique()
print('species:', df['species'].unique())    # 종의 고유한 값 종류
print('island :', df['island'].unique())     # 서식지의 고유한 값 종류
print('sex    :', df['sex'].unique())        # 성별의 고유한 값 종류
print()
print("성별이 '.'인 행")
df[df['sex'] == '.']

In [ ]:
#@title ④ 성별 이상치 행 삭제하기
print('삭제할 행의 인덱스:', df[df['sex'] == '.'].index.tolist())
df.drop(df[df['sex'] == '.'].index, axis=0, inplace=True)   # 교과서: df.drop(336, axis=0, inplace=True)
print('남은 성별 값:', df['sex'].unique())
print('데이터 크기:', df.shape)

In [ ]:
#@title ⑤ 데이터 정규화하기: 최소-최대 정규화 (교과서 82쪽)
from sklearn.preprocessing import MinMaxScaler
num_cols = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']

df_raw = df.copy()                         # 정규화 전 데이터를 따로 남겨 두기
scaler = MinMaxScaler()                    # MinMaxScaler 생성하기
df[num_cols] = scaler.fit_transform(df[num_cols])   # 수치형 속성값을 0~1로 바꾸기
df.head(10)

In [ ]:
#@title 정규화 전과 후 비교하기 (교과서 76쪽)
attribute = "flipper_length_mm" #@param ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sns.histplot(df_raw[attribute], bins=20, kde=True, ax=axes[0]); axes[0].set_title('before')
sns.histplot(df[attribute], bins=20, kde=True, ax=axes[1], color='green'); axes[1].set_title('after (0~1)')
plt.show()
print(f'정규화 전: {df_raw[attribute].min()} ~ {df_raw[attribute].max()}  →  정규화 후: {df[attribute].min():.1f} ~ {df[attribute].max():.1f}')

In [ ]:
#@title 단계 4 핵심 속성 추출하기: 산점도 (교과서 82쪽)
x_attr = "culmen_length_mm" #@param ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
y_attr = "culmen_depth_mm" #@param ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
sns.scatterplot(x=x_attr, y=y_attr, data=df, hue='species')
plt.show()

**해 보기**: 교과서처럼 ① 부리 길이 × 부리 깊이, ② 부리 깊이 × 날개 길이로 바꿔 실행해 보세요. 어떤 조합이 세 종을 더 잘 나누나요?

In [ ]:
#@title 상관계수 히트맵 (교과서 83쪽)
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='Greens')
plt.show()

In [ ]:
#@title 산점도와 회귀선: 부리 길이와 체질량 (교과서 83쪽)
sns.regplot(x='culmen_length_mm', y='body_mass_g', data=df, color='green')
plt.show()
r = df['culmen_length_mm'].corr(df['body_mass_g'])
print(f'부리 길이와 체질량의 상관계수: {r:.2f}')

### 정리하기
1. 결측치와 이상치를 처리한 뒤 데이터는 몇 개가 되었나요?
2. 체질량과 상관계수가 가장 큰 속성은 무엇인가요? 그 값은?
3. 펭귄의 **종**을 구분하는 데 가장 좋은 두 속성의 조합은 무엇이라고 생각하나요? 근거를 산점도로 설명해 보세요.